# Stage 1 — Continued Pre-Training (CPT), done efficiently

Adapts `google/gemma-3-1b-it` to Swahili so it stops drifting to other
languages and represents Swahili morphology better, **before** any
translation supervision.

### What changed vs. the previous attempt
| Problem before | Fix here |
|---|---|
| `bf16=True` on a T4 (no bf16 tensor cores) | **fp16** everywhere — uses the T4's fp16 tensor cores |
| `attn_implementation="eager"` (slowest) | **SDPA** attention (FA2 isn't available on Turing) |
| Per-example padding wasted most compute | **Token packing** into fixed 1024-blocks — ~zero padding |
| Data shrunk to 40K rows to fit 12h | **Checkpoint + resume to the HF Hub** → train the FULL corpus across as many sessions as needed |
| CPT claimed to fix embeddings but LoRA never touched them | **`modules_to_save=["embed_tokens"]`** → the embeddings actually adapt |

> **Honest note:** CPT does *not* change the tokenizer, so raw *tokens/word*
> will not drop. What improves is the model's internal representation of
> Swahili and its tendency to stay in Swahili. That is the real benefit.

### How to run across the 12h wall
Set `HF_USERNAME`, then run top-to-bottom. If Kaggle kills the session at
12h, just start a **new** session and run it again — it auto-resumes from
the last checkpoint on the Hub. When the epoch finishes, the final cell
merges everything into a standalone model that Stage 2 consumes.

> **Kaggle:** Settings → Accelerator → **GPU P100** (simplest, one card) or
> **GPU T4 ×2** (this notebook uses one card; DDP is an optional ~1.8× on top).
> Internet **ON**. Add your `HF_TOKEN` as a Kaggle Secret.

In [ ]:
import os
# Single GPU by default (no distributed code = nothing to debug).
# To use BOTH T4s with DDP instead: remove this line and launch the script
# form with:  accelerate launch --multi_gpu --num_processes 2 train.py
# DDP is a ~1.8x speedup ON TOP of the fixes below — those matter more.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Reduce fragmentation OOMs (must be set before torch initializes CUDA):
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%%capture
!pip install -q sacrebleu bitsandbytes

In [ ]:
import torch, transformers, peft, platform
print("Python:", platform.python_version(), "| torch:", torch.__version__,
      "| transformers:", transformers.__version__, "| peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} ({p.total_memory/1e9:.1f} GB)")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("HF auth OK")

## Configuration

In [ ]:
# ---- Model ----
MODEL_ID = "google/gemma-3-1b-it"

# ---- Hub (REQUIRED for resume-across-sessions) ----
HF_USERNAME    = "Dev-Noble"                       # <-- EDIT THIS
HUB_CPT_CKPT   = f"{HF_USERNAME}/swa-gemma-cpt"           # rolling checkpoints
HUB_CPT_MERGED = f"{HF_USERNAME}/swa-gemma-cpt-merged"    # final model -> Stage 2

# ---- Data (resume lifts the 12h limit, so use a lot; None = full split) ----
JAMII_ROWS  = 200_000      # of ~979k available
INKUBA_ROWS = 200_000
MIN_CHARS   = 30
BLOCK_SIZE  = 512          # packed sequence length (512 is T4-safe with Gemma's 262k vocab)

# ---- Training ----
LR, LR_SCHED   = 2e-5, "cosine"
NUM_EPOCHS     = 1
PER_DEV_BS     = 2         # T4-safe; raise if you have headroom
GRAD_ACCUM     = 16        # effective batch = 32
WARMUP_RATIO   = 0.03
WEIGHT_DECAY   = 0.01
MAX_GRAD_NORM  = 1.0
SAVE_STEPS     = 200       # checkpoint often -> safe to be killed at 12h
LOG_STEPS      = 20

# ---- LoRA ----
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05
LORA_TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
# Training embeddings deepens Swahili adaptation but is memory-heavy on a
# 15GB T4 (Gemma's embedding matrix is large). OFF by default so the run
# fits. To enable: set ["embed_tokens"] AND drop PER_DEV_BS to 1.
TRAIN_EMBEDDINGS = []

OUTPUT_DIR = "/kaggle/working/cpt_out"
os.makedirs(OUTPUT_DIR, exist_ok=True)
assert HF_USERNAME != "YOUR_HF_USERNAME", "Set HF_USERNAME first."
print("effective batch:", PER_DEV_BS * GRAD_ACCUM, "| block:", BLOCK_SIZE)

## Tokenizer

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print("vocab size:", tok.vocab_size, "| pad:", repr(tok.pad_token))

## Load data and **pack** it

Tokenize with no padding, concatenate everything, then slice into equal
1024-token blocks. Every token trained on is real signal — this is the big
throughput win for CPT on short forum text.

In [ ]:
from datasets import load_dataset, concatenate_datasets, Value, Features
from itertools import chain

def load_slice(name, cfg=None, n=None, split="train"):
    sp = f"{split}[:{n}]" if n else split
    return load_dataset(name, cfg, split=sp) if cfg else load_dataset(name, split=sp)

print("Loading JamiiForums...")
jamii = load_slice("AlexLeoTz/jamiiforums-2b", n=JAMII_ROWS)

print("Loading Inkuba-Mono (sw)...")
try:
    inkuba = load_slice("lelapa/Inkuba-Mono", cfg="swahili", n=INKUBA_ROWS)
except Exception as e:
    print("  lelapa failed (", e, ") -> Alfaxad mirror")
    inkuba = load_slice("Alfaxad/Inkuba-Mono-Swahili", n=INKUBA_ROWS)

def to_text(ds):
    if "text" in ds.column_names:
        ds = ds.select_columns(["text"])
    else:
        col = next(c for c in ds.column_names if "string" in str(ds.features[c]).lower())
        ds = ds.rename_column(col, "text").select_columns(["text"])
    return ds.cast(Features({"text": Value("string")}))

raw = concatenate_datasets([to_text(jamii), to_text(inkuba)]).shuffle(seed=42)
raw = raw.filter(lambda x: x["text"] and len(x["text"].strip()) >= MIN_CHARS)
print("raw rows:", len(raw))

tokd = raw.map(lambda b: tok(b["text"]), batched=True,
               remove_columns=raw.column_names, desc="tokenize")

def group(ex):
    ids = list(chain(*ex["input_ids"]))
    total = (len(ids) // BLOCK_SIZE) * BLOCK_SIZE
    ids = ids[:total]
    blocks = [ids[i:i+BLOCK_SIZE] for i in range(0, total, BLOCK_SIZE)]
    return {"input_ids": blocks,
            "attention_mask": [[1]*BLOCK_SIZE for _ in blocks],
            "labels": [b[:] for b in blocks]}

packed = tokd.map(group, batched=True, desc="pack")
print("packed blocks:", len(packed), "| ~tokens:", len(packed)*BLOCK_SIZE)

## Load base model — 4-bit, **fp16 compute, SDPA attention**

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16, NOT bf16 (T4 = no bf16 tensor cores)
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map={"": 0},
    torch_dtype=torch.float16,
    attn_implementation="sdpa",             # sdpa, NOT eager
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
print("VRAM after model load:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

## Apply LoRA (+ trainable embeddings)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    modules_to_save=(TRAIN_EMBEDDINGS or None),
    bias="none",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()
print("VRAM after LoRA:", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

## Resume from the Hub if a previous session left a checkpoint

In [ ]:
from huggingface_hub import create_repo, list_repo_files, snapshot_download

def fetch_resume(repo_id, local="/kaggle/working/resume_ckpt"):
    """Download the 'last-checkpoint' pushed by a previous session, if any."""
    try:
        files = list_repo_files(repo_id, repo_type="model")
    except Exception:
        print("No checkpoint repo yet -> starting fresh.")
        return None
    if any(f.startswith("last-checkpoint/") for f in files):
        snapshot_download(repo_id, allow_patterns="last-checkpoint/*",
                          local_dir=local, repo_type="model")
        ckpt = os.path.join(local, "last-checkpoint")
        print("Resuming from:", ckpt)
        return ckpt
    print("Repo exists but no last-checkpoint -> starting fresh.")
    return None

create_repo(HUB_CPT_CKPT, private=True, exist_ok=True)
resume_ckpt = fetch_resume(HUB_CPT_CKPT)

## Train (pushes `last-checkpoint` to the Hub every save)

In [ ]:
from transformers import TrainingArguments, Trainer, default_data_collator

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEV_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, lr_scheduler_type=LR_SCHED,
    warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM,
    fp16=True, bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=LOG_STEPS,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=2,
    report_to="none", dataloader_num_workers=2, dataloader_pin_memory=False,
    gradient_checkpointing=False,   # enabled on the model already
    push_to_hub=True, hub_model_id=HUB_CPT_CKPT,
    hub_strategy="checkpoint", hub_private_repo=True,
)

trainer = Trainer(model=model, args=args, train_dataset=packed,
                  data_collator=default_data_collator)
trainer.train(resume_from_checkpoint=resume_ckpt)
print("training loss:", round(trainer.state.log_history[-1].get("train_loss", float("nan")), 4)
      if trainer.state.log_history else "n/a")

## Merge into a standalone Swahili-adapted model (input to Stage 2)

Reload the base in fp16, apply the trained adapter, merge, push. The result
is a normal model — no adapter juggling in Stage 2.

In [ ]:
import gc
from peft import PeftModel

adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
trainer.model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)

del model, trainer
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                            device_map={"": 0})
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
merged.push_to_hub(HUB_CPT_MERGED, private=True)
tok.push_to_hub(HUB_CPT_MERGED)
print("Merged CPT model pushed to:", HUB_CPT_MERGED)

## Smoke test — does it produce Swahili (beam search, not sampling)?

In [ ]:
merged.config.use_cache = True
merged.eval()

def translate(en, n=64):
    p = ("<start_of_turn>user\nTranslate the following English sentence to Swahili.\n"
         f"English: {en}\nSwahili:<end_of_turn>\n<start_of_turn>model\n")
    enc = tok(p, return_tensors="pt").to(merged.device)
    out = merged.generate(**enc, max_new_tokens=n, num_beams=5, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for s in ["The children are playing outside.",
          "The government will build a new hospital next year."]:
    print(s, "\n  ->", translate(s), "\n")